# load summary reports

In [2]:
from __future__ import annotations

import json
import re
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs pandas. Select the project .venv kernel, "
        "or install dependencies with: uv sync"
    ) from exc

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")



# Set to None to load every model folder under logs/.
# Examples:
# MODEL_FILTER = "qwen3.6-plus"
# MODEL_FILTER = ["qwen3.6-plus", "gpt5.5"]
MODEL_FILTER = None

# Set True if you want CSV copies of the DataFrames written to analysis/outputs/.
EXPORT_CSV = False

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "analysis" else Path(".").resolve()
LOGS_ROOT = PROJECT_ROOT / "logs"
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "outputs"

assert LOGS_ROOT.exists(), f"Logs folder not found: {LOGS_ROOT}"


def normalise_model_filter(model_filter):
    if model_filter is None:
        return None
    if isinstance(model_filter, str):
        return {model_filter}
    return set(model_filter)


def discover_report_paths(logs_root: Path, model_filter=None) -> list[Path]:
    allowed_models = normalise_model_filter(model_filter)
    report_paths = []

    for model_dir in sorted(p for p in logs_root.iterdir() if p.is_dir()):
        if allowed_models is not None and model_dir.name not in allowed_models:
            continue

        for run_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
            report_path = run_dir / "task_run_report.json"
            if report_path.exists():
                report_paths.append(report_path)

    return report_paths


report_paths = discover_report_paths(LOGS_ROOT, MODEL_FILTER)
print(f"Found {len(report_paths)} reports")
for path in report_paths[:10]:
    print(path.relative_to(PROJECT_ROOT))


def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def leading_run_index(run_name: str):
    match = re.match(r"^(\d+)", run_name)
    return int(match.group(1)) if match else None


reports = []
for report_path in report_paths:
    run_dir = report_path.parent
    model_dir = run_dir.parent
    report = read_json(report_path)
    reports.append({
        "model_name": model_dir.name,
        "run_name": run_dir.name,
        "run_index": leading_run_index(run_dir.name),
        "report_path": str(report_path.relative_to(PROJECT_ROOT)),
        "report": report,
    })

len(reports)


report_rows = []
task_rows = []
tool_score_rows = []

for item in reports:
    report = item["report"]
    summary = report.get("summary", {}) or {}
    base = {
        "model_name": item["model_name"],
        "run_name": item["run_name"],
        "run_index": item["run_index"],
        "report_path": item["report_path"],
        "generated_at": report.get("generated_at"),
        "source_csv": report.get("source_csv"),
    }

    report_rows.append({**base, **summary})

    for result in report.get("results", []) or []:
        validation = result.get("ground_truth_validation") or {}
        query_trace = result.get("query_trace") or {}
        tool_appropriateness = result.get("tool_appropriateness") or {}

        task_base = {
            **base,
            "task_id": result.get("task_id"),
            "prompt": result.get("prompt"),
            "status": result.get("status"),
            "answer_type": result.get("answer_type"),
            "passed": validation.get("passed", result.get("passed")),
            "ground_truth_score": validation.get("score"),
            "validation_strategy": validation.get("strategy"),
            "parameter_schema_valid_rate": result.get("parameter_schema_valid_rate"),
            "constraint_compliance_rate": result.get("constraint_compliance_rate"),
            "tool_appropriateness": tool_appropriateness.get("average_score"),
            "tool_appropriateness_weighted": tool_appropriateness.get("weighted_score"),
            "total_tokens": result.get("total_tokens"),
            "input_tokens": query_trace.get("input_tokens"),
            "output_tokens": query_trace.get("output_tokens"),
            "total_steps": result.get("total_steps"),
            "total_tool_calls": result.get("total_tool_calls"),
            "latency": result.get("latency"),
            "end_to_end_latency": result.get("end_to_end_latency"),
            "attempt_count": result.get("attempt_count"),
            "tools_used": result.get("tools_used") or query_trace.get("tools_used"),
            "final_answer": query_trace.get("final_answer", result.get("final_answer")),
        }
        task_rows.append(task_base)

        for call_index, scored_call in enumerate(tool_appropriateness.get("scored_calls") or [], start=1):
            tool_score_rows.append({
                **base,
                "task_id": result.get("task_id"),
                "call_index": call_index,
                **scored_call,
            })

reports_df = pd.DataFrame(report_rows)
task_runs_df = pd.DataFrame(task_rows)
tool_scores_df = pd.DataFrame(tool_score_rows)

if not task_runs_df.empty:
    task_runs_df["passed"] = task_runs_df["passed"].astype("boolean")
    numeric_columns = [
        "ground_truth_score",
        "parameter_schema_valid_rate",
        "constraint_compliance_rate",
        "tool_appropriateness",
        "tool_appropriateness_weighted",
        "total_tokens",
        "input_tokens",
        "output_tokens",
        "total_steps",
        "total_tool_calls",
        "latency",
        "end_to_end_latency",
        "attempt_count",
    ]
    for col in numeric_columns:
        if col in task_runs_df.columns:
            task_runs_df[col] = pd.to_numeric(task_runs_df[col], errors="coerce")

if not tool_scores_df.empty and "score" in tool_scores_df.columns:
    tool_scores_df["score"] = pd.to_numeric(tool_scores_df["score"], errors="coerce")

print(f"reports_df: {reports_df.shape[0]} rows x {reports_df.shape[1]} columns")
print(f"task_runs_df: {task_runs_df.shape[0]} rows x {task_runs_df.shape[1]} columns")
print(f"tool_scores_df: {tool_scores_df.shape[0]} rows x {tool_scores_df.shape[1]} columns")








Found 18 reports
logs\claude-optus-4.6\1anthropic_claude-opus-4.6_20260512T082634Z\task_run_report.json
logs\claude-optus-4.6\2anthropic_claude-opus-4.6_20260512T084016Z\task_run_report.json
logs\claude-optus-4.6\3anthropic_claude-opus-4.6_20260516T090351Z\task_run_report.json
logs\claude-sonnet-4.6\1anthropic_claude-sonnet-4.6_20260512T072817Z\task_run_report.json
logs\claude-sonnet-4.6\2anthropic_claude-sonnet-4.6_20260512T073036Z\task_run_report.json
logs\claude-sonnet-4.6\3anthropic_claude-sonnet-4.6_20260512T073516Z\task_run_report.json
logs\gpt5.5\openai_gpt-5.5_20260512T075029Z\task_run_report.json
logs\gpt5.5\openai_gpt-5.5_20260512T075044Z\task_run_report.json
logs\gpt5.5\openai_gpt-5.5_20260512T075850Z\task_run_report.json
logs\kimi-k2.6\1moonshotai_kimi-k2.6_20260512T065725Z\task_run_report.json
reports_df: 18 rows x 28 columns
task_runs_df: 630 rows x 27 columns
tool_scores_df: 3106 rows x 12 columns


## Task-Level Performance Exploration

This section excludes `llama4` because its runs are unstable, then ranks tasks by task completion rate (TCR). Lower TCR means more model-task failures across the remaining models/runs.

In [ ]:
EXCLUDED_MODELS = {"llama4"}
WORST_N = 15

TASK_CATEGORY_LABELS = {
    "A": "Discovery & Dataset Overview",
    "B": "Filtered Counting & Lookup",
    "C": "Aggregation & Temporal Pattern Analysis",
    "D": "Comparative Analysis",
    "E": "Exploratory NLP Analysis",
}

task_runs_for_task_analysis = task_runs_df[~task_runs_df["model_name"].isin(EXCLUDED_MODELS)].copy()
task_runs_for_task_analysis["task_category"] = task_runs_for_task_analysis["task_id"].astype(str).str.extract(r"^([A-Za-z])", expand=False).str.upper()
task_runs_for_task_analysis["task_category_name"] = task_runs_for_task_analysis["task_category"].map(TASK_CATEGORY_LABELS).fillna("Unknown")

task_tcr = task_runs_for_task_analysis.groupby(
    ["task_id", "task_category", "task_category_name"],
    as_index=False,
).agg(
    prompt=("prompt", "first"),
    models=("model_name", "nunique"),
    runs=("run_name", "nunique"),
    task_runs=("task_id", "count"),
    successful_task_runs=("passed", "sum"),
    task_completion_rate=("passed", "mean"),
    average_latency=("latency", "mean"),
    average_tokens=("total_tokens", "mean"),
)

task_tcr["failed_task_runs"] = task_tcr["task_runs"] - task_tcr["successful_task_runs"]

worst_task_tcr = task_tcr.sort_values(
    ["task_completion_rate", "failed_task_runs", "task_id"],
    ascending=[True, False, True],
)

print(f"Excluded models: {sorted(EXCLUDED_MODELS)}")
print(f"Analysed models: {sorted(task_runs_for_task_analysis['model_name'].unique())}")
display(worst_task_tcr.head(WORST_N))

### Worst Tasks by Model

The pivot below shows each model's TCR for the worst tasks. This helps distinguish globally hard tasks from tasks that are mostly caused by one model.

In [ ]:
worst_task_ids = worst_task_tcr.head(WORST_N)["task_id"].tolist()

task_model_tcr = task_runs_for_task_analysis.groupby(
    ["task_id", "model_name"],
    as_index=False,
).agg(task_completion_rate=("passed", "mean"))

task_model_tcr_matrix = task_model_tcr[task_model_tcr["task_id"].isin(worst_task_ids)].pivot_table(
    index="task_id",
    columns="model_name",
    values="task_completion_rate",
)

task_model_tcr_matrix = task_model_tcr_matrix.reindex(worst_task_ids)
task_model_tcr_matrix_percent = task_model_tcr_matrix.map(lambda value: f"{value:.2%}" if pd.notna(value) else "")
display(task_model_tcr_matrix_percent)